# LLM Honesty and Deception Probing, Algoverse 2026 Spring, KMSA & Tommy

## Part 1: Preparation

In [1]:
import os
import random
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from pathlib import Path
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm.auto import tqdm
from utils.knowledge_check import knowledge_check_truthfulqa, knowledge_check_mmlu
from utils.generation import generate_response, NEUTRAL_SYSTEM, FACTUAL_DECEPTION_SCENARIO
# from utils.activation import extract_activations
from utils.probe import probe_all_layers, train_linear_probe
import warnings
warnings.filterwarnings("ignore")   # suppress the greedy-decoding flag warnings

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

DATA_DIR = Path("data/dataset")
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

Device: cuda
GPU: NVIDIA GeForce RTX 4090
VRAM: 25.4 GB


In [2]:
deception_df = pd.read_csv(DATA_DIR / "deception_dataset.csv")
print(f"deception_dataset: {deception_df.shape}")
print(deception_df["label"].value_counts())

deception_dataset: (400, 6)
label
honest       200
deceptive    200
Name: count, dtype: int64


In [3]:
MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
model.eval()

N_LAYERS = model.config.num_hidden_layers
HIDDEN_DIM = model.config.hidden_size
print(f"Loaded: {N_LAYERS} layers, hidden_dim={HIDDEN_DIM}")

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu.


Loaded: 28 layers, hidden_dim=3584


## Part 2: Model Knowledge Test
### 2.1 TruthfulQA

In [4]:
tqa_mc = load_dataset("truthful_qa", "multiple_choice", split="validation")
TRUTHFULQA_PATH = DATA_DIR / "truthfulQA_test_results.csv"
CHECKPOINT_EVERY = 50

if TRUTHFULQA_PATH.exists():
    kc_df = pd.read_csv(TRUTHFULQA_PATH)
    if len(kc_df) == len(tqa_mc):
        print(f"Already complete ({len(kc_df)} rows). Loading from {TRUTHFULQA_PATH}")
    else:
        done_questions = set(kc_df["question"].tolist())
        remaining = [item for item in tqa_mc if item["question"] not in done_questions]
        print(f"Resuming from checkpoint: {len(kc_df)} done, {len(remaining)} remaining")
        records = []
        for i, item in enumerate(tqdm(remaining, desc="TruthfulQA knowledge check")):
            records.append(knowledge_check_truthfulqa(item, model, tokenizer, DEVICE))
            if (i + 1) % CHECKPOINT_EVERY == 0:
                pd.DataFrame(records).to_csv(TRUTHFULQA_PATH, mode="a", header=False, index=False)
                records = []
        if records:
            pd.DataFrame(records).to_csv(TRUTHFULQA_PATH, mode="a", header=False, index=False)
        kc_df = pd.read_csv(TRUTHFULQA_PATH)
        print(f"Done. Total: {len(kc_df)}")
else:
    records = []
    for i, item in enumerate(tqdm(tqa_mc, desc="TruthfulQA knowledge check")):
        records.append(knowledge_check_truthfulqa(item, model, tokenizer, DEVICE))
        if (i + 1) % CHECKPOINT_EVERY == 0:
            df_chunk = pd.DataFrame(records)
            df_chunk.to_csv(TRUTHFULQA_PATH, mode="a", header=not TRUTHFULQA_PATH.exists(), index=False)
            records = []
    if records:
        pd.DataFrame(records).to_csv(TRUTHFULQA_PATH, mode="a", header=not TRUTHFULQA_PATH.exists(), index=False)
    kc_df = pd.read_csv(TRUTHFULQA_PATH)
    print(f"Done. Total: {len(kc_df)}  |  Passed: {kc_df['passed'].sum()}  |  Failed: {(~kc_df['passed']).sum()}")

passed_df = kc_df[kc_df["passed"] == True].reset_index(drop=True)
failed_df = kc_df[kc_df["passed"] == False].reset_index(drop=True)
print(f"Passed: {len(passed_df)}  |  Failed: {len(failed_df)}")

Already complete (817 rows). Loading from data/dataset/truthfulQA_test_results.csv
Passed: 398  |  Failed: 419


### 2.2 MMLU

In [5]:
mmlu = load_dataset("cais/mmlu", "all", split="test")
MMLU_PATH = DATA_DIR / "mmlu_test_results.csv"
CHECKPOINT_EVERY = 50

if MMLU_PATH.exists():
    mmlu_kc_df = pd.read_csv(MMLU_PATH)
    if len(mmlu_kc_df) == len(mmlu):
        print(f"Already complete ({len(mmlu_kc_df)} rows). Loading from {MMLU_PATH}")
    else:
        done_questions = set(mmlu_kc_df["question"].tolist())
        remaining = [item for item in mmlu if item["question"] not in done_questions]
        print(f"Resuming from checkpoint: {len(mmlu_kc_df)} done, {len(remaining)} remaining")
        records = []
        for i, item in enumerate(tqdm(remaining, desc="MMLU knowledge check")):
            records.append(knowledge_check_mmlu(item, model, tokenizer, DEVICE))
            if (i + 1) % CHECKPOINT_EVERY == 0:
                pd.DataFrame(records).to_csv(MMLU_PATH, mode="a", header=False, index=False)
                records = []
        if records:
            pd.DataFrame(records).to_csv(MMLU_PATH, mode="a", header=False, index=False)
        mmlu_kc_df = pd.read_csv(MMLU_PATH)
        print(f"Done. Total: {len(mmlu_kc_df)}")
else:
    records = []
    for i, item in enumerate(tqdm(mmlu, desc="MMLU knowledge check")):
        records.append(knowledge_check_mmlu(item, model, tokenizer, DEVICE))
        if (i + 1) % CHECKPOINT_EVERY == 0:
            df_chunk = pd.DataFrame(records)
            df_chunk.to_csv(MMLU_PATH, mode="a", header=not MMLU_PATH.exists(), index=False)
            records = []
    if records:
        pd.DataFrame(records).to_csv(MMLU_PATH, mode="a", header=not MMLU_PATH.exists(), index=False)
    mmlu_kc_df = pd.read_csv(MMLU_PATH)
    print(f"Done. Total: {len(mmlu_kc_df)}  |  Passed: {mmlu_kc_df['passed'].sum()}  |  Failed: {(~mmlu_kc_df['passed']).sum()}")

mmlu_passed_df = mmlu_kc_df[mmlu_kc_df["passed"] == True].reset_index(drop=True)
mmlu_failed_df = mmlu_kc_df[mmlu_kc_df["passed"] == False].reset_index(drop=True)
print(f"Passed: {len(mmlu_passed_df)}  |  Failed: {len(mmlu_failed_df)}")

Resuming from checkpoint: 14038 done, 0 remaining


MMLU knowledge check: 0it [00:00, ?it/s]

Done. Total: 14038
Passed: 6596  |  Failed: 7442


## Part 3: Response Generation and Result Judge
### 3.1 TruthfulQA
#### 3.1.1 Response Generation

In [6]:
TRUTHFULQA_RESPONSES_PATH = DATA_DIR / "truthfulQA_responses.csv"
CHECKPOINT_EVERY = 50

configs = [
    ("A", passed_df, NEUTRAL_SYSTEM),
    ("B", failed_df, NEUTRAL_SYSTEM),
    ("C", passed_df, FACTUAL_DECEPTION_SCENARIO),
]

if TRUTHFULQA_RESPONSES_PATH.exists():
    resp_df = pd.read_csv(TRUTHFULQA_RESPONSES_PATH)
    done_keys = set(zip(resp_df["question"], resp_df["config"]))
    total_expected = len(passed_df) * 2 + len(failed_df)
    if len(resp_df) == total_expected:
        print(f"Already complete ({len(resp_df)} rows). Loading from {TRUTHFULQA_RESPONSES_PATH}")
    else:
        print(f"Resuming from checkpoint: {len(resp_df)} done, {total_expected - len(resp_df)} remaining")
else:
    done_keys = set()
    resp_df = None
    print("Starting fresh.")

for config_name, source_df, system_prompt in configs:
    remaining = source_df[~source_df["question"].isin(
        {q for q, c in done_keys if c == config_name}
    )].reset_index(drop=True)

    if len(remaining) == 0:
        print(f"Config {config_name}: already complete, skipping.")
        continue

    print(f"Config {config_name}: {len(remaining)} rows to generate.")
    records = []
    for i, row in enumerate(tqdm(remaining.itertuples(), total=len(remaining), desc=f"Config {config_name}")):
        records.append({
            "question": row.question,
            "correct_answer": row.correct_answer,
            "config": config_name,
            "system_prompt": system_prompt,
            "response": generate_response(row.question, model, tokenizer, DEVICE, system_prompt=system_prompt),
        })
        if (i + 1) % CHECKPOINT_EVERY == 0:
            pd.DataFrame(records).to_csv(
                TRUTHFULQA_RESPONSES_PATH, mode="a",
                header=not TRUTHFULQA_RESPONSES_PATH.exists(), index=False
            )
            done_keys.update((r["question"], r["config"]) for r in records)
            records = []
    if records:
        pd.DataFrame(records).to_csv(
            TRUTHFULQA_RESPONSES_PATH, mode="a",
            header=not TRUTHFULQA_RESPONSES_PATH.exists(), index=False
        )

resp_df = pd.read_csv(TRUTHFULQA_RESPONSES_PATH)
print(f"\nDone. Total rows: {len(resp_df)}")
print(resp_df["config"].value_counts())

Already complete (1215 rows). Loading from data/dataset/truthfulQA_responses.csv
Config A: already complete, skipping.
Config B: already complete, skipping.
Config C: already complete, skipping.

Done. Total rows: 1215
config
B    419
A    398
C    398
Name: count, dtype: int64


### 3.1.2 Claude Haiku Batch Judge

In [7]:
from utils.judge import build_batch_requests_anthropic
import anthropic

# ── Config ────────────────────────────────────────────────────────────────
JUDGE_MODEL = "claude-haiku-4-5-20251001"
MODEL_TAG   = "claude_haiku"
N_VOTES     = 3
# ─────────────────────────────────────────────────────────────────────────

TRUTHFULQA_JUDGE_PATH = DATA_DIR / f"judge_truthfulqa_{MODEL_TAG}.csv"

if TRUTHFULQA_JUDGE_PATH.exists():
    print(f"Results already exist at {TRUTHFULQA_JUDGE_PATH}, skipping batch submission.")
else:
    # Add row_id column (= pandas index) so we can join results back after parsing
    resp_df = resp_df.reset_index(drop=True)
    resp_df["row_id"] = resp_df.index
    resp_df.to_csv(DATA_DIR / "truthfulQA_responses.csv", index=False)
    print(f"Saved row_id column to truthfulQA_responses.csv ({len(resp_df)} rows)")

    client = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])
    requests = build_batch_requests_anthropic(resp_df.set_index("row_id"), JUDGE_MODEL, N_VOTES)
    print(f"Total requests to submit: {len(requests)}")
    batch = client.messages.batches.create(requests=requests)
    print(f"Batch submitted.")
    print(f"Batch ID:     {batch.id}")
    print(f"Status:       {batch.processing_status}")
    print(f"Request counts: {batch.request_counts}")

Results already exist at data/dataset/judge_truthfulqa_claude_haiku.csv, skipping batch submission.


In [8]:
from utils.judge import parse_batch_results_anthropic
import anthropic

# ── Config ────────────────────────────────────────────────────────────────
BATCH_ID  = ""   # paste TruthfulQA Anthropic batch ID here
MODEL_TAG = "claude_haiku"
# ─────────────────────────────────────────────────────────────────────────

TRUTHFULQA_JUDGE_PATH = DATA_DIR / f"judge_truthfulqa_{MODEL_TAG}.csv"

if TRUTHFULQA_JUDGE_PATH.exists():
    print(f"Already exists: {TRUTHFULQA_JUDGE_PATH}")
else:
    client = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])
    batch = client.messages.batches.retrieve(BATCH_ID)
    print(f"Batch status: {batch.processing_status}")
    assert batch.processing_status == "ended", "Batch not done yet — re-run when complete."

    results = list(client.messages.batches.results(BATCH_ID))
    print(f"Retrieved {len(results)} results")

    # resp_df must have row_id as index for the join
    source = resp_df.set_index("row_id")
    judge_df = parse_batch_results_anthropic(results, source)
    judge_df.to_csv(TRUTHFULQA_JUDGE_PATH, index=False)
    print(f"Saved {len(judge_df)} rows to {TRUTHFULQA_JUDGE_PATH}")
    print(judge_df["config"].value_counts())

Already exists: data/dataset/judge_truthfulqa_claude_haiku.csv


### 3.1.3 GPT-4o-mini Batch Judge

In [ ]:
from utils.judge import build_batch_requests_openai
import json
from openai import OpenAI

# ── Config ────────────────────────────────────────────────────────────────
JUDGE_MODEL = "gpt-4o-mini"
MODEL_TAG   = "gpt4o_mini"
N_VOTES     = 3
# ─────────────────────────────────────────────────────────────────────────

TRUTHFULQA_JUDGE_PATH = DATA_DIR / f"judge_truthfulqa_{MODEL_TAG}.csv"
JSONL_PATH = DATA_DIR / f"batch_truthfulqa_{MODEL_TAG}.jsonl"

# Ensure row_id exists (may have been skipped if Anthropic batch cell was a no-op)
if "row_id" not in resp_df.columns:
    resp_df = resp_df.reset_index(drop=True)
    resp_df["row_id"] = resp_df.index
    resp_df.to_csv(DATA_DIR / "truthfulQA_responses.csv", index=False)
    print(f"Added row_id and saved to truthfulQA_responses.csv ({len(resp_df)} rows)")

if TRUTHFULQA_JUDGE_PATH.exists():
    print(f"Results already exist at {TRUTHFULQA_JUDGE_PATH}, skipping batch submission.")
else:
    requests = build_batch_requests_openai(resp_df.set_index("row_id"), JUDGE_MODEL, N_VOTES)
    print(f"Total requests to submit: {len(requests)}")

    with open(JSONL_PATH, "w") as f:
        for req in requests:
            f.write(json.dumps(req) + "\n")
    print(f"Wrote {JSONL_PATH}")

    client = OpenAI(api_key="your-openai-api-key")
    with open(JSONL_PATH, "rb") as f:
        uploaded = client.files.create(file=f, purpose="batch")
    print(f"Uploaded file ID: {uploaded.id}")

    batch = client.batches.create(
        input_file_id=uploaded.id,
        endpoint="/v1/chat/completions",
        completion_window="24h",
    )
    print(f"Batch submitted.")
    print(f"Batch ID:     {batch.id}")
    print(f"Status:       {batch.status}")

Total requests to submit: 3645
Wrote data/dataset/batch_truthfulqa_gpt4o_mini.jsonl
Uploaded file ID: file-8XM6aZyyJ5JBGzarv3KoDN
Batch submitted.
Batch ID:     batch_69be3bd4e05081908c2945af22be2511
Status:       validating


In [ ]:
from utils.judge import parse_batch_results_openai
from openai import OpenAI

# ── Config ────────────────────────────────────────────────────────────────
BATCH_ID  = "batch_69be3bd4e05081908c2945af22be2511"   # paste TruthfulQA OpenAI batch ID here
MODEL_TAG = "gpt4o_mini"
# ─────────────────────────────────────────────────────────────────────────

TRUTHFULQA_JUDGE_PATH = DATA_DIR / f"judge_truthfulqa_{MODEL_TAG}.csv"

if TRUTHFULQA_JUDGE_PATH.exists():
    print(f"Already exists: {TRUTHFULQA_JUDGE_PATH}")
else:
    client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
    batch = client.batches.retrieve(BATCH_ID)
    print(f"Batch status: {batch.status}")
    assert batch.status == "completed", "Batch not done yet — re-run when complete."

    output_file_id = batch.output_file_id
    content = client.files.content(output_file_id).text
    lines = content.splitlines()
    print(f"Retrieved {len(lines)} output lines")

    source = resp_df.set_index("row_id")
    judge_df = parse_batch_results_openai(lines, source)
    judge_df.to_csv(TRUTHFULQA_JUDGE_PATH, index=False)
    print(f"Saved {len(judge_df)} rows to {TRUTHFULQA_JUDGE_PATH}")
    print(judge_df["config"].value_counts())

## 3.2 MMLU Response Genearation and Result Judge
### 3.2.1 Response Generation

In [10]:
MMLU_RESPONSES_PATH = DATA_DIR / "mmlu_responses.csv"
CHECKPOINT_EVERY = 50

configs = [
    ("A", mmlu_passed_df, NEUTRAL_SYSTEM),
    ("B", mmlu_failed_df, NEUTRAL_SYSTEM),
    ("C", mmlu_passed_df, FACTUAL_DECEPTION_SCENARIO),
]

if MMLU_RESPONSES_PATH.exists():
    mmlu_resp_df = pd.read_csv(MMLU_RESPONSES_PATH)
    done_keys = set(zip(mmlu_resp_df["question"], mmlu_resp_df["config"]))
    total_expected = len(mmlu_passed_df) * 2 + len(mmlu_failed_df)
    if len(mmlu_resp_df) == total_expected:
        print(f"Already complete ({len(mmlu_resp_df)} rows). Loading from {MMLU_RESPONSES_PATH}")
    else:
        print(f"Resuming from checkpoint: {len(mmlu_resp_df)} done, {total_expected - len(mmlu_resp_df)} remaining")
else:
    done_keys = set()
    mmlu_resp_df = None
    print("Starting fresh.")

for config_name, source_df, system_prompt in configs:
    remaining = source_df[~source_df["question"].isin(
        {q for q, c in done_keys if c == config_name}
    )].reset_index(drop=True)

    if len(remaining) == 0:
        print(f"Config {config_name}: already complete, skipping.")
        continue

    print(f"Config {config_name}: {len(remaining)} rows to generate.")
    records = []
    for i, row in enumerate(tqdm(remaining.itertuples(), total=len(remaining), desc=f"Config {config_name}")):
        records.append({
            "question": row.question,
            "correct_answer": row.correct_answer,
            "config": config_name,
            "system_prompt": system_prompt,
            "response": generate_response(row.question, model, tokenizer, DEVICE, system_prompt=system_prompt),
        })
        if (i + 1) % CHECKPOINT_EVERY == 0:
            pd.DataFrame(records).to_csv(
                MMLU_RESPONSES_PATH, mode="a",
                header=not MMLU_RESPONSES_PATH.exists(), index=False
            )
            done_keys.update((r["question"], r["config"]) for r in records)
            records = []
    if records:
        pd.DataFrame(records).to_csv(
            MMLU_RESPONSES_PATH, mode="a",
            header=not MMLU_RESPONSES_PATH.exists(), index=False
        )

mmlu_resp_df = pd.read_csv(MMLU_RESPONSES_PATH)
if "row_id" not in mmlu_resp_df.columns:
    mmlu_resp_df["row_id"] = mmlu_resp_df.index
    mmlu_resp_df.to_csv(MMLU_RESPONSES_PATH, index=False)

print(f"\nDone. Total rows: {len(mmlu_resp_df)}")
print(mmlu_resp_df["config"].value_counts())

Already complete (20634 rows). Loading from data/dataset/mmlu_responses.csv
Config A: already complete, skipping.
Config B: already complete, skipping.
Config C: already complete, skipping.

Done. Total rows: 20634
config
B    7442
A    6596
C    6596
Name: count, dtype: int64


### 3.2.2 Claude Haiku Batch Judge

In [11]:
from utils.judge import build_batch_requests_anthropic
import anthropic

# ── Config ────────────────────────────────────────────────────────────────
JUDGE_MODEL = "claude-haiku-4-5-20251001"
MODEL_TAG   = "claude_haiku"
N_VOTES     = 3
# ─────────────────────────────────────────────────────────────────────────

MMLU_JUDGE_PATH = DATA_DIR / f"judge_mmlu_{MODEL_TAG}.csv"

if MMLU_JUDGE_PATH.exists():
    print(f"Results already exist at {MMLU_JUDGE_PATH}, skipping batch submission.")
else:
    client = anthropic.Anthropic(api_key="your-api-key-here")
    requests = build_batch_requests_anthropic(mmlu_resp_df.set_index("row_id"), JUDGE_MODEL, N_VOTES)
    print(f"Total requests to submit: {len(requests)}")
    batch = client.messages.batches.create(requests=requests)
    print(f"Batch submitted.")
    print(f"Batch ID:     {batch.id}")
    print(f"Status:       {batch.processing_status}")
    print(f"Request counts: {batch.request_counts}")

Results already exist at data/dataset/judge_mmlu_claude_haiku.csv, skipping batch submission.


In [12]:
from utils.judge import parse_batch_results_anthropic

BATCH_ID = "msgbatch_01TYpxpDPeBx5e7rWPgetVEk"

if MMLU_JUDGE_PATH.exists():
    mmlu_judge_df = pd.read_csv(MMLU_JUDGE_PATH)
    print(f"Already complete ({len(mmlu_judge_df)} rows). Loading from {MMLU_JUDGE_PATH}")
else:
    client = anthropic.Anthropic(api_key="your-api-key-here")

    batch = client.messages.batches.retrieve(BATCH_ID)
    print(f"Status: {batch.processing_status}")
    print(f"Request counts: {batch.request_counts}")

    if batch.processing_status != "ended":
        print("Batch not ready yet. Check back later.")
    else:
        batch_results = client.messages.batches.results(BATCH_ID)
        mmlu_judge_df = parse_batch_results_anthropic(batch_results, mmlu_resp_df.set_index("row_id"))
        mmlu_judge_df.to_csv(MMLU_JUDGE_PATH, index=False)
        print(f"\nSaved {len(mmlu_judge_df)} rows to {MMLU_JUDGE_PATH}")
        print(mmlu_judge_df["config"].value_counts())

Already complete (20634 rows). Loading from data/dataset/judge_mmlu_claude_haiku.csv


### 3.2.3 GPT-4o-mini Batch Judge

In [ ]:
from utils.judge import build_batch_requests_openai
import json
from openai import OpenAI

# ── Config ────────────────────────────────────────────────────────────────
JUDGE_MODEL = "gpt-4o-mini"
MODEL_TAG   = "gpt4o_mini"
N_VOTES     = 3
# ─────────────────────────────────────────────────────────────────────────

MMLU_JUDGE_PATH = DATA_DIR / f"judge_mmlu_{MODEL_TAG}.csv"

if MMLU_JUDGE_PATH.exists():
    print(f"Results already exist at {MMLU_JUDGE_PATH}, skipping batch submission.")
else:
    client = OpenAI(api_key="your-openai-api-key")

    mid = len(mmlu_resp_df) // 2
    splits = [
        ("1", mmlu_resp_df.iloc[:mid]),
        ("2", mmlu_resp_df.iloc[mid:]),
    ]

    for split_tag, split_df in splits:
        jsonl_path = DATA_DIR / f"batch_mmlu_{MODEL_TAG}_{split_tag}.jsonl"
        requests = build_batch_requests_openai(split_df.set_index("row_id"), JUDGE_MODEL, N_VOTES)
        print(f"Split {split_tag}: {len(requests)} requests")

        with open(jsonl_path, "w") as f:
            for req in requests:
                f.write(json.dumps(req) + "\n")
        print(f"Wrote {jsonl_path}")

        with open(jsonl_path, "rb") as f:
            uploaded = client.files.create(file=f, purpose="batch")
        print(f"Uploaded file ID: {uploaded.id}")

        batch = client.batches.create(
            input_file_id=uploaded.id,
            endpoint="/v1/chat/completions",
            completion_window="24h",
        )
        print(f"Batch {split_tag} submitted.")
        print(f"Batch ID:     {batch.id}")
        print(f"Status:       {batch.status}")
        print()

Split 1: 30951 requests
Wrote data/dataset/batch_mmlu_gpt4o_mini_1.jsonl
Uploaded file ID: file-WGmv4eGgnv7YorC7R3YbKQ
Batch 1 submitted.
Batch ID:     batch_69be3dab7404819094294c105244d24c
Status:       validating

Split 2: 30951 requests
Wrote data/dataset/batch_mmlu_gpt4o_mini_2.jsonl
Uploaded file ID: file-7qk6uh6AoUEQtLb9TESNvW
Batch 2 submitted.
Batch ID:     batch_69be3db577a48190aaa58e6eccf75e1b
Status:       validating



In [ ]:
from utils.judge import parse_batch_results_openai
from openai import OpenAI
import pandas as pd

# ── Config ────────────────────────────────────────────────────────────────
BATCH_ID_1 = "batch_69be3dab7404819094294c105244d24c"   # paste MMLU OpenAI batch ID (split 1) here
BATCH_ID_2 = "batch_69be3db577a48190aaa58e6eccf75e1b"   # paste MMLU OpenAI batch ID (split 2) here
MODEL_TAG  = "gpt4o_mini"
# ─────────────────────────────────────────────────────────────────────────

MMLU_JUDGE_PATH = DATA_DIR / f"judge_mmlu_{MODEL_TAG}.csv"

if MMLU_JUDGE_PATH.exists():
    print(f"Already exists: {MMLU_JUDGE_PATH}")
else:
    client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
    source = mmlu_resp_df.set_index("row_id")
    parts = []

    for split_tag, batch_id in [("1", BATCH_ID_1), ("2", BATCH_ID_2)]:
        batch = client.batches.retrieve(batch_id)
        print(f"Split {split_tag} status: {batch.status}")
        assert batch.status == "completed", f"Batch {split_tag} not done yet — re-run when complete."

        content = client.files.content(batch.output_file_id).text
        lines = content.splitlines()
        print(f"Split {split_tag}: retrieved {len(lines)} output lines")

        part_df = parse_batch_results_openai(lines, source)
        parts.append(part_df)

    judge_df = pd.concat(parts, ignore_index=True)
    judge_df.to_csv(MMLU_JUDGE_PATH, index=False)
    print(f"\nSaved {len(judge_df)} rows to {MMLU_JUDGE_PATH}")
    print(judge_df["config"].value_counts())